# Analyse d'erreurs & interprétabilité de modèles

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Téléchargement des données

In [ ]:
!git clone https://github.com/shuuchuu/dataset-landscape.git

## Import de TensorFlow et des autres librairies nécessaires

In [ ]:
!pip install shap
import itertools
import os
import pathlib
import random
import typing

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL
import seaborn
import shap
import skimage.transform
import sklearn.utils
import sklearn.metrics
import tensorflow as tf
import tensorflow.keras as keras
import tqdm.notebook

## Préparation des données

Pour charger nos données, nous allons combiner plusieurs libraires : [OpenCV](https://opencv.org/), [NumPy](https://numpy.org/) et [scikit-learn](https://scikit-learn.org/stable/). Ces librairies seront appelées depuis la fonction `get_images`.

Après avoir chargé chaque image, nous allons passer leur canaux en RGB puis les redimensionner à 150x150, enfin, par défaut, nous retournerons un dataset mélangé grâce à [`sklearn.utils.shuffle`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.shuffle.html).

In [ ]:
INPUT_SHAPE = (150, 150)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> typing.Tuple[tf.Tensor, tf.Tensor]:
  images = []
  if create_labels:
    labels = []

  # On itère sur les sous-dossier de la racine : ils correspondent chacun à une
  # classe
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Traitement des dossiers"):

    dir_name = subdir_path.name

    if create_labels:
      # On attribue le bon label en fonction du nom du dossier "labels"
      label = label_to_index.get(dir_name)

    # On ajoute chaque image du label (dossier) courant à notre dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False):
      # Utilisation de PIL pour charger l'image
      images.append(
          numpy.array(PIL.Image.open(image_path).resize(INPUT_SHAPE)))
      if create_labels:
        labels.append(label)

  images = tf.constant(numpy.array(images))
  if create_labels:
    labels = tf.constant(numpy.array(labels))

  if shuffle:
    perm = tf.random.shuffle(tf.range(images.shape[0]))
    images = tf.gather(images, perm)
    if create_labels:
      labels = tf.gather(labels, perm)

  if create_labels:
    return images, labels
  else:
    return images

## Appel à `get_images`

In [ ]:
images, labels = get_images(pathlib.Path("dataset-landscape") / "seg_train")

In [ ]:
print(f"Forme des images : {images.shape}")
print(f"Forme des labels : {labels.shape}")

seaborn.countplot(x=labels.numpy())
plt.title("Décomptes des différents labels")
plt.ylabel("Décompte")
plt.xlabel("Label")
plt.show()

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[labels[img_index]]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision par
    # ordinateur
    ax[i, j].imshow(image)
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Entraînement d'un modèle peaufiné (*finetuné*)

In [ ]:
base_model = keras.applications.EfficientNetB0(include_top=False,
                                               weights="imagenet",
                                               input_shape=(150, 150, 3))

base_model.trainable = False

transferred_cnn = keras.Sequential(
    [base_model,
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(1024, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(256, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(64, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(16, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Flatten(),
     keras.layers.Dense(6, activation="softmax", kernel_regularizer="l2")],
    name="transferred_cnn")

transferred_cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                        loss="sparse_categorical_crossentropy",
                        metrics=["accuracy"])

transferred_cnn.summary()

In [ ]:
training = transferred_cnn.fit(images,
                               labels,
                               epochs=5,
                               validation_split=0.30,
                               batch_size=512)

## Évaluation des performances sur l'ensemble de test

Dans le dossier `seg_test` se trouve un ensemble de données qui n'ont jamais été vues durant l'apprentissage.

On utilisera la méthode `evaluate(X, y)` du modèle pour évaluer la qualité de nos prédictions sur ce dataset.

In [ ]:
test_images,test_labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_test")

In [ ]:
transferred_cnn.evaluate(test_images, test_labels)

## Analyse d'erreur

On affiche la matrice de confusion puis on regarde des images mal classées.

In [ ]:
def predict(model: keras.Model, images: tf.Tensor) -> tf.Tensor:
  return tf.math.argmax(model.predict(images), axis=-1)


def analyze_preds(preds: tf.Tensor, labels: tf.Tensor) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names,
                  annot=True,
                  fmt="d")
  plt.title("Matrice de confusion")
  plt.show()

  seaborn.countplot(x=[label_names[x] for x in preds])
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Class")
  plt.show()


test_preds = predict(transferred_cnn, test_images)
analyze_preds(test_preds, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str,
                  true_class: str,
                  images: tf.Tensor,
                  preds: tf.Tensor,
                  labels: tf.Tensor
                  ) -> None:
  print(f"Prédiction : {predicted_class}, classe réelle : {true_class}")
  mistakes = images[(preds == label_to_index[predicted_class])
                         & (labels == label_to_index[true_class])]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  if mistakes.shape[1] == 3:
    mistakes = tf.transpose(mistakes, perm=(0, 2, 3, 1))

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(mistakes[img_index])
    ax[i, j].axis("off")
  plt.show()

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label montagne
plot_mistakes("glacier", "mountain", test_images, test_preds, test_labels)

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label mer
plot_mistakes("glacier", "sea", test_images, test_preds, test_labels)

In [ ]:
# Plot les images prédites bâtiment alors qu'elles ont un label mer
plot_mistakes("buildings", "sea", test_images, test_preds, test_labels)

In [ ]:
def evaluate(model: keras.Model, images: tf.Tensor, labels: tf.Tensor) -> None:
  loss, accuracy = model.evaluate(images, labels, verbose=False)
  print(f"Loss: {loss:.2f}, accuracy: {accuracy:.2f}")
  preds = predict(model, images)
  analyze_preds(preds, labels)
  plot_mistakes("glacier", "mountain", images, preds, labels)
  plot_mistakes("glacier", "sea", images, preds, labels)
  plot_mistakes("buildings", "sea", images, preds, labels)


evaluate(transferred_cnn, test_images, test_labels)

## Prédire dans des condition « réelles »

Dans le dossier `seg_pred` se trouvent des images non-annotées. On ne peut donc pas évaluer correctement les performances sur cet ensemble.

Cependant, on peut afficher des photos et les probabilités que notre modèle attribue à chaque classe.

In [ ]:
pred_images = get_images(
    pathlib.Path("dataset-landscape") / "seg_pred",
    create_labels=False)

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Récupération de l'image et prédiction de sa classe
    image = pred_images[img_index]
    probabilities = transferred_cnn.predict(image[None, ...])[0]
    predicted_class = label_names[numpy.argmax(probabilities)]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision
    # par ordinateur
    ax[i * 2, j].imshow(image)
    ax[i * 2, j].set_title(f"Exemple {img_index}")
    ax[i * 2, j].axis('off')

    # Affichage de la distribution de prédiction sur la ligne d'en dessous
    ax[i * 2 + 1, j].bar(label_names, probabilities)

## Interprétabilité de modèle avec SHAP

Utilisez la bibliothèque `shap` pour interpréter les prédictions du modèle sur les 10 premiers exemples du jeu de données d'entraînement (`images`).

In [ ]:
!pip install shap

In [ ]:
# Votre code ici

### Solution

In [ ]:
import shap

masker = shap.maskers.Image("blur(32,32)", images[0].shape)

explainer = shap.Explainer(transferred_cnn.predict, masker, output_names=label_names)

shap_values = explainer(
    images[:10].numpy(), max_evals=1_000, batch_size=50, outputs=shap.Explanation.argsort.flip[:3]
)

In [ ]:
shap.image_plot(shap_values)

## SHAP sur des données structurées

Commençons par récupérer des données et entraîner un modèle&nbsp;:

In [ ]:
#@title Récupération des données et fonction de prétraitement
!git clone https://github.com/nzmonzmp/dataset-ames.git
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import sklearn.ensemble
import sklearn.model_selection
import scipy.stats
import warnings
warnings.filterwarnings('ignore')

def preprocess(train_file, test_file):
    train_X = pandas.read_csv(train_file, index_col="Id")
    test_X = pandas.read_csv(test_file, index_col="Id")

    train_y = train_X.pop("SalePrice")

    all_X = pandas.concat([train_X, test_X])

    # Fill with median
    cols_1 = ["LotFrontage"]
    all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

    # Fill with mode
    cols_2 = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st",
             "Exterior2nd", "SaleType", "Utilities"]
    all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

    # Fill with 0
    cols_4 = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
              "BsmtFinSF2", "BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF",
              "MasVnrArea", "TotalBsmtSF"]
    all_X[cols_4] = all_X[cols_4].fillna(0)

    # Other fills
    cols_5 = ["Functional"]
    all_X[cols_5] = all_X[cols_5].fillna("Typ")

    # On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
    all_X = all_X.fillna("NA")

    # On transforme le codage numérique en string afin que ce soit traité comme
    # une variable catégorielle
    cols_numerical2label = ['MSSubClass']
    all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

    quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
    quality_columns = ["BsmtCond", "BsmtQual", "ExterCond", "ExterQual",
                       "FireplaceQu", "GarageCond", "GarageQual", "HeatingQC",
                       "KitchenQual", "PoolQC"]
    street_mapping = dict(NA=0, Grvl=1, Pave=2)
    bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

    replace_mapping = dict(
      Alley=street_mapping,
      BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
      BsmtFinType1=bsmt_fin_mapping,
      BsmtFinType2=bsmt_fin_mapping,
      Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
      LandSlope=dict(Sev=1, Mod=2, Gtl=3),
      LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
      PavedDrive=dict(NA=0, N=1, P=2, Y=3),
      Street=dict(Grvl=1, Pave=2),
      Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
    )

    for quality_column in quality_columns:
      replace_mapping[quality_column] = quality_mapping

    all_X.replace(replace_mapping, inplace=True)

    print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

    dummies = pandas.get_dummies(all_X)
    return (dummies.iloc[:train_X.shape[0], :],
            train_y,
            dummies.iloc[train_X.shape[0]:, :])

In [ ]:
X, y, _ = preprocess("dataset-ames/train.csv", "dataset-ames/test.csv")

In [ ]:
rfr = sklearn.ensemble.RandomForestRegressor()
rfr.fit(X, y)

## Explication globale

Utilisez la bibliothèque `shap` pour calculer l'apport marginal de chaque caractéristique sur l'ensemble du jeu de données d'entraînement.

In [ ]:
# Votre code ici

### Solution

In [ ]:
explainer = shap.TreeExplainer(rfr)
explanation = explainer(X)

In [ ]:
shap.plots.bar(explanation)

## Explication locale

Affichez maintenant le premier exemple du jeu de données et son explication.

In [ ]:
# Votre code ici

### Solution

In [ ]:
X.head(1)

In [ ]:
shap.plots.bar(explanation[0])

In [ ]:
shap.plots.force(explainer.expected_value, explanation[0].values, matplotlib=True, feature_names=X.columns)